<a href="https://colab.research.google.com/github/ThisumiWijesinghe/Fraud-Detection-with-Federated-Learning/blob/main/fedbn_and_avg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================
# PaySim Fraud Detection using DNNs (No FL, FedAvg, FedBN)
# ================================================

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# -----------------------------
# 1. Load and preprocess dataset
# -----------------------------
# Load PaySim dataset (replace path if needed)
import kagglehub

# Download dataset from Kaggle
path = kagglehub.dataset_download("ealaxi/paysim1")

# Define file path
file_path = path + "/PS_20174392719_1491204439457_log.csv"

# Load dataset
df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
# Encode categorical 'type'
le = LabelEncoder()
df["type"] = le.fit_transform(df["type"])

# Select features and labels
features = ["step", "type", "amount",
            "oldbalanceOrg", "newbalanceOrig",
            "oldbalanceDest", "newbalanceDest"]

X = df[features].values
y = df["isFraud"].values

INPUT_DIM = X.shape[1]

# -----------------------------
# 2. Split data
# -----------------------------
NUM_CLIENTS = 12
alpha = 0.5  # Dirichlet parameter
TEST_SIZE = 0.2

def dirichlet_split(X, y, num_clients, alpha):
    """Split training data among clients using Dirichlet distribution (non-IID)."""
    data_per_client = [[] for _ in range(num_clients)]
    labels = np.unique(y)
    for label in labels:
        idx = np.where(y == label)[0]
        np.random.shuffle(idx)
        proportions = np.random.dirichlet(np.repeat(alpha, num_clients))
        proportions = (np.cumsum(proportions) * len(idx)).astype(int)[:-1]
        split_idx = np.split(idx, proportions)
        for i in range(num_clients):
            data_per_client[i].extend(split_idx[i])
    clients_data = {}
    for i in range(num_clients):
        client_idx = data_per_client[i]
        clients_data[i] = {"X": X[client_idx], "y": y[client_idx]}
    return clients_data

# Split dataset into train/test normally
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=42, stratify=y)

# Split training data among clients (non-IID)
raw_clients_train = dirichlet_split(X_train_full, y_train_full, NUM_CLIENTS, alpha)

# -----------------------------
# 3. Prepare final client data
# -----------------------------
clients_train_data = {}
clients_test_data = {}

for i in range(NUM_CLIENTS):
    # Client training data
    X_c = raw_clients_train[i]["X"]
    y_c = raw_clients_train[i]["y"]
    clients_train_data[i] = {
        "X": X_c.astype('float32'),
        "y": y_c.astype('float32')
    }

    # Client test data (same for all, normally split)
    clients_test_data[i] = {
        "X": X_test_full.astype('float32'),
        "y": y_test_full.astype('float32')
    }

# Print sample distribution
for cid in range(NUM_CLIENTS):
    Xc, yc = clients_train_data[cid]["X"], clients_train_data[cid]["y"]
    total = len(yc)
    fraud = np.sum(yc)
    nonfraud = total - fraud
    print(f"Client {cid+1}: Total={total}, Fraud={fraud}, Non-Fraud={nonfraud}, Fraud%={fraud/total*100:.4f}")

# -----------------------------
# 4. Define DNN model
# -----------------------------
def create_model(input_dim=INPUT_DIM):
    model = tf.keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu"),
        layers.BatchNormalization(),
        layers.Dense(32, activation="relu"),
        layers.BatchNormalization(),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

# -----------------------------
# 5. Local training (No FL)
# -----------------------------
LOCAL_EPOCHS = 5
BATCH_SIZE = 2048

print("\n--- Training DNNs Locally (No FL) ---")
client_models_noFL = []

for cid in range(NUM_CLIENTS):
    print(f"Training Client {cid+1} locally...")
    model = create_model()
    model.fit(
        clients_train_data[cid]["X"], clients_train_data[cid]["y"],
        epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE, verbose=1
    )
    client_models_noFL.append(model)

# Evaluate local models
print("\n--- Local Model Accuracy ---")
local_accs = []
for cid, model in enumerate(client_models_noFL):
    loss, acc = model.evaluate(clients_test_data[cid]["X"], clients_test_data[cid]["y"], verbose=0)
    local_accs.append(acc)
    print(f"Client {cid+1} Accuracy: {acc:.4f}")
print(f"Average Local Accuracy: {np.mean(local_accs):.4f}")

# -----------------------------
# 6. FedAvg Training
# -----------------------------
NUM_ROUNDS = 20

def fedavg_aggregate(client_weights):
    new_weights = []
    for weights in zip(*client_weights):
        new_weights.append(np.mean(weights, axis=0))
    return new_weights

print("\n--- Starting FedAvg Training ---")
global_model_fedavg = create_model()

for r in range(NUM_ROUNDS):
    print(f"\n--- Global Round {r+1}/{NUM_ROUNDS} ---")
    client_weights = []

    for cid in range(NUM_CLIENTS):
        local_model = create_model()
        local_model.set_weights(global_model_fedavg.get_weights())
        print(f" Client {cid+1} training...")
        local_model.fit(
            clients_train_data[cid]["X"], clients_train_data[cid]["y"],
            epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE, verbose=1
        )
        client_weights.append(local_model.get_weights())

    global_model_fedavg.set_weights(fedavg_aggregate(client_weights))
    print(f"Round {r+1} aggregation complete.")

# Evaluate FedAvg model
print("\n--- FedAvg Global Model Accuracy ---")
fedavg_accs = []
for cid in range(NUM_CLIENTS):
    loss, acc = global_model_fedavg.evaluate(clients_test_data[cid]["X"], clients_test_data[cid]["y"], verbose=0)
    fedavg_accs.append(acc)
    print(f"Client {cid+1} Accuracy: {acc:.4f}")
print(f"Average FedAvg Accuracy: {np.mean(fedavg_accs):.4f}")

# -----------------------------
# 7. FedBN Training
# -----------------------------
print("\n--- Starting FedBN Training ---")

# Initialize global and client models
global_model_fedbn = create_model()
client_models_fedbn = [create_model() for _ in range(NUM_CLIENTS)]
for m in client_models_fedbn:
    m.set_weights(global_model_fedbn.get_weights())

def sync_non_bn(global_model, local_model):
    for g_layer, l_layer in zip(global_model.layers, local_model.layers):
        if 'batch_normalization' not in g_layer.name.lower():
            l_layer.set_weights(g_layer.get_weights())

def fedbn_aggregate(client_models, global_model):
    new_weights = []
    for i, layer in enumerate(global_model.layers):
        if 'batch_normalization' not in layer.name.lower():
            weights_all_clients = [m.layers[i].get_weights() for m in client_models]
            avg_weights = [np.mean(w, axis=0) for w in zip(*weights_all_clients)]
            new_weights.extend(avg_weights)
        else:
            new_weights.extend(layer.get_weights())
    return new_weights

for r in range(NUM_ROUNDS):
    print(f"\n--- Global Round {r+1}/{NUM_ROUNDS} ---")
    for cid in range(NUM_CLIENTS):
        print(f" Client {cid+1} training...")
        local_model = client_models_fedbn[cid]
        sync_non_bn(global_model_fedbn, local_model)
        local_model.fit(
            clients_train_data[cid]["X"], clients_train_data[cid]["y"],
            epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE, verbose=1
        )
    global_model_fedbn.set_weights(fedbn_aggregate(client_models_fedbn, global_model_fedbn))

# Evaluate FedBN models
print("\n--- FedBN Client Models Accuracy ---")
fedbn_accs = []
for cid in range(NUM_CLIENTS):
    loss, acc = client_models_fedbn[cid].evaluate(clients_test_data[cid]["X"], clients_test_data[cid]["y"], verbose=0)
    fedbn_accs.append(acc)
    print(f"Client {cid+1} Accuracy: {acc:.4f}")
print(f"Average FedBN Accuracy: {np.mean(fedbn_accs):.4f}")

Streaming output truncated to the last 5000 lines.
Epoch 3/5
616/616 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.9999 - loss: 3.7030e-04
Epoch 4/5
616/616 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.9999 - loss: 3.0168e-04
Epoch 5/5
616/616 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.9999 - loss: 3.0644e-04
 Client 2 training...
Epoch 1/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9984 - loss: 0.0063
Epoch 2/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9983 - loss: 0.0068
Epoch 3/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9986 - loss: 0.0060
Epoch 4/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9986 - loss: 0.0054
Epoch 5/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9987 - loss: 0.0051
 Client 3 training...
Epoch 1/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.8046 - loss: 0.9878
Epoch 2/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8346 - loss: 0.8257
Epoch 3/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/st

In [2]:
# ================================================
# PaySim Fraud Detection using DNNs (No FL, FedAvg, FedBN)
# ================================================

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# -----------------------------
# 1. Load and preprocess dataset
# -----------------------------
# Load PaySim dataset (replace path if needed)
import kagglehub

# Download dataset from Kaggle
path = kagglehub.dataset_download("ealaxi/paysim1")

# Define file path
file_path = path + "/PS_20174392719_1491204439457_log.csv"

# Load dataset
df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
# Encode categorical 'type'
le = LabelEncoder()
df["type"] = le.fit_transform(df["type"])

# Select features and labels
features = ["step", "type", "amount",
            "oldbalanceOrg", "newbalanceOrig",
            "oldbalanceDest", "newbalanceDest"]

X = df[features].values
y = df["isFraud"].values

INPUT_DIM = X.shape[1]

# -----------------------------
# 2. Split data
# -----------------------------
NUM_CLIENTS = 12
alpha = 0.5  # Dirichlet parameter
TEST_SIZE = 0.2

def dirichlet_split(X, y, num_clients, alpha):
    """Split training data among clients using Dirichlet distribution (non-IID)."""
    data_per_client = [[] for _ in range(num_clients)]
    labels = np.unique(y)
    for label in labels:
        idx = np.where(y == label)[0]
        np.random.shuffle(idx)
        proportions = np.random.dirichlet(np.repeat(alpha, num_clients))
        proportions = (np.cumsum(proportions) * len(idx)).astype(int)[:-1]
        split_idx = np.split(idx, proportions)
        for i in range(num_clients):
            data_per_client[i].extend(split_idx[i])
    clients_data = {}
    for i in range(num_clients):
        client_idx = data_per_client[i]
        clients_data[i] = {"X": X[client_idx], "y": y[client_idx]}
    return clients_data

# Split dataset into train/test normally
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=42, stratify=y)

# Split training data among clients (non-IID)
raw_clients_train = dirichlet_split(X_train_full, y_train_full, NUM_CLIENTS, alpha)

# -----------------------------
# 3. Prepare final client data
# -----------------------------
clients_train_data = {}
clients_test_data = {}

for i in range(NUM_CLIENTS):
    # Client training data
    X_c = raw_clients_train[i]["X"]
    y_c = raw_clients_train[i]["y"]
    clients_train_data[i] = {
        "X": X_c.astype('float32'),
        "y": y_c.astype('float32')
    }

    # Client test data (same for all, normally split)
    clients_test_data[i] = {
        "X": X_test_full.astype('float32'),
        "y": y_test_full.astype('float32')
    }

# Print sample distribution
for cid in range(NUM_CLIENTS):
    Xc, yc = clients_train_data[cid]["X"], clients_train_data[cid]["y"]
    total = len(yc)
    fraud = np.sum(yc)
    nonfraud = total - fraud
    print(f"Client {cid+1}: Total={total}, Fraud={fraud}, Non-Fraud={nonfraud}, Fraud%={fraud/total*100:.4f}")

# -----------------------------
# 4. Define DNN model
# -----------------------------
def create_model(input_dim=INPUT_DIM):
    model = tf.keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu"),
        layers.BatchNormalization(),
        layers.Dense(32, activation="relu"),
        layers.BatchNormalization(),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

Using Colab cache for faster access to the 'paysim1' dataset.
Dataset shape: (6362620, 11)
Client 1: Total=232932, Fraud=747.0, Non-Fraud=232185.0, Fraud%=0.3207
Client 2: Total=48746, Fraud=31.0, Non-Fraud=48715.0, Fraud%=0.0636
Client 3: Total=1198484, Fraud=909.0, Non-Fraud=1197575.0, Fraud%=0.0758
Client 4: Total=92959, Fraud=12.0, Non-Fraud=92947.0, Fraud%=0.0129
Client 5: Total=933813, Fraud=501.0, Non-Fraud=933312.0, Fraud%=0.0537
Client 6: Total=875687, Fraud=1026.0, Non-Fraud=874661.0, Fraud%=0.1172
Client 7: Total=9521, Fraud=289.0, Non-Fraud=9232.0, Fraud%=3.0354
Client 8: Total=177396, Fraud=561.0, Non-Fraud=176835.0, Fraud%=0.3162
Client 9: Total=1002236, Fraud=195.0, Non-Fraud=1002041.0, Fraud%=0.0195
Client 10: Total=141838, Fraud=1656.0, Non-Fraud=140182.0, Fraud%=1.1675
Client 11: Total=276205, Fraud=268.0, Non-Fraud=275937.0, Fraud%=0.0970
Client 12: Total=100279, Fraud=375.0, Non-Fraud=99904.0, Fraud%=0.3740


In [ ]:
print("\n==============================")
print("TRAINING WITHOUT FEDERATED LEARNING")
print("==============================")

LOCAL_EPOCHS = 5
BATCH_SIZE = 2048

client_models_noFL = []

for cid in range(NUM_CLIENTS):
    print(f"\nClient {cid+1} Local Training")
    model = create_model()
    model.fit(
        clients_train_data[cid]["X"], clients_train_data[cid]["y"],
        epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE, verbose=1
    )
    client_models_noFL.append(model)

print("\n--- Local Model Accuracy ---")
local_accs = []
for cid, model in enumerate(client_models_noFL):
    loss, acc = model.evaluate(
        clients_test_data[cid]["X"],
        clients_test_data[cid]["y"],
        verbose=0
    )
    local_accs.append(acc)
    print(f"Client {cid+1} Accuracy: {acc:.4f}")

print(f"\n✅ Average Local Accuracy: {np.mean(local_accs):.4f}")


TRAINING WITHOUT FEDERATED LEARNING

Client 1 Local Training
Epoch 1/5
114/114 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9238 - loss: 0.4824
Epoch 2/5
114/114 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.9970 - loss: 0.1563
Epoch 3/5
114/114 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.9981 - loss: 0.0539
Epoch 4/5
114/114 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9982 - loss: 0.0280
Epoch 5/5
114/114 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.9983 - loss: 0.0189

Client 2 Local Training
Epoch 1/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.7243 - loss: 0.6718
Epoch 2/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9684 - loss: 0.5640
Epoch 3/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9936 - loss: 0.4710
Epoch 4/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9978 - loss: 0.3785
Epoch 5/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9991 - loss: 0.2932

Client 3 Local Training
Epoch 1/5
586/586 ━━━━━━━━━━━━━━━━━━━━

In [ ]:
print("\n==============================")
print("FEDERATED LEARNING WITH FedAvg")
print("==============================")

NUM_ROUNDS = 20
round_accs_fedavg = []

global_model_fedavg = create_model()

def fedavg_aggregate(client_weights):
    new_weights = []
    for weights in zip(*client_weights):
        new_weights.append(np.mean(weights, axis=0))
    return new_weights

for r in range(NUM_ROUNDS):
    print(f"\n Global Round {r+1}/{NUM_ROUNDS}")

    client_weights = []

    # Train all clients silently
    for cid in range(NUM_CLIENTS):
        local_model = create_model()
        local_model.set_weights(global_model_fedavg.get_weights())

        local_model.fit(
            clients_train_data[cid]["X"],
            clients_train_data[cid]["y"],
            epochs=LOCAL_EPOCHS,
            batch_size=BATCH_SIZE,
            verbose=0
        )

        client_weights.append(local_model.get_weights())

    # Server aggregation
    global_model_fedavg.set_weights(fedavg_aggregate(client_weights))

    # Evaluate global model
    accs = []
    for cid in range(NUM_CLIENTS):
        loss, acc = global_model_fedavg.evaluate(
            clients_test_data[cid]["X"],
            clients_test_data[cid]["y"],
            verbose=0
        )
        accs.append(acc)

    avg_acc = np.mean(accs)
    round_accs_fedavg.append(avg_acc)
    print(f" Round {r+1} Average Accuracy: {avg_acc:.4f}")

print(f"\n Final FedAvg Accuracy: {round_accs_fedavg[-1]:.4f}")

In [ ]:
print("\n==============================")
print("FEDERATED LEARNING WITH FedBN")
print("==============================")

round_accs_fedbn = []

global_model_fedbn = create_model()
client_models_fedbn = [create_model() for _ in range(NUM_CLIENTS)]

for m in client_models_fedbn:
    m.set_weights(global_model_fedbn.get_weights())

def sync_non_bn(global_model, local_model):
    for g_layer, l_layer in zip(global_model.layers, local_model.layers):
        if 'batch_normalization' not in g_layer.name.lower():
            l_layer.set_weights(g_layer.get_weights())

def fedbn_aggregate(client_models, global_model):
    new_weights = []
    for i, layer in enumerate(global_model.layers):
        if 'batch_normalization' not in layer.name.lower():
            weights_all_clients = [m.layers[i].get_weights() for m in client_models]
            avg_weights = [np.mean(w, axis=0) for w in zip(*weights_all_clients)]
            new_weights.extend(avg_weights)
        else:
            new_weights.extend(layer.get_weights())
    return new_weights

for r in range(NUM_ROUNDS):
    print(f"\n🌍 Global Round {r+1}/{NUM_ROUNDS}")

    # Train all clients silently
    for cid in range(NUM_CLIENTS):
        local_model = client_models_fedbn[cid]
        sync_non_bn(global_model_fedbn, local_model)

        local_model.fit(
            clients_train_data[cid]["X"],
            clients_train_data[cid]["y"],
            epochs=LOCAL_EPOCHS,
            batch_size=BATCH_SIZE,
            verbose=0
        )

    # Aggregate
    global_model_fedbn.set_weights(
        fedbn_aggregate(client_models_fedbn, global_model_fedbn)
    )

    # Evaluate client models
    accs = []
    for cid in range(NUM_CLIENTS):
        loss, acc = client_models_fedbn[cid].evaluate(
            clients_test_data[cid]["X"],
            clients_test_data[cid]["y"],
            verbose=0
        )
        accs.append(acc)

    avg_acc = np.mean(accs)
    round_accs_fedbn.append(avg_acc)
    print(f" Round {r+1} Average Accuracy: {avg_acc:.4f}")

print(f"\n Final FedBN Accuracy: {round_accs_fedbn[-1]:.4f}")